In [1]:
# ---------------------------------------------------------------------------
# Common setup and model (enlarged-seeding model of Sections 2-3).
# Fibers are thin cylinders of radius r; midpoints are seeded on the enlarged
# region S = [-lmax/2, 1+lmax/2]^3 and contacts are counted inside the unit
# cube C via the contact-point gate. Grayscale seaborn figures, 600 dpi,
# PNG + PDF, no titles/captions. No paths are printed (kept anonymous).
# ---------------------------------------------------------------------------
import os, json
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"font.size": 10, "axes.edgecolor": "0.3", "savefig.dpi": 600})
GRAY, DARK = "0.45", "0.15"

_FIG = os.path.join("..", "results", "figures")
_TAB = os.path.join("..", "results", "tables")
_DAT = os.path.join("..", "data")
for _d in (_FIG, _TAB, _DAT):
    os.makedirs(_d, exist_ok=True)

def savefig(fig, name):
    """Save a figure as PNG and PDF (600 dpi, no title); do not print paths."""
    fig.savefig(os.path.join(_FIG, name + ".png"), bbox_inches="tight")
    fig.savefig(os.path.join(_FIG, name + ".pdf"), bbox_inches="tight")
    plt.close(fig)

def save_table(df, name, float_format="%.4f"):
    df.to_latex(os.path.join(_TAB, name + ".tex"), index=False, float_format=float_format)

def save_csv(df, name):
    df.to_csv(os.path.join(_DAT, name + ".csv"), index=False)

# ---- model constants (two-length mixture) --------------------------------
ELL = np.array([0.2, 0.4]); PMIX = np.array([0.5, 0.5])
LMAX = float(ELL.max()); LBAR = float((PMIX * ELL).sum())
CVOL, SVOL = 1.0, (1.0 + LMAX) ** 3        # |C| = 1, |S| = 1.4^3

def raw_moment(k):
    return float((PMIX * ELL ** k).sum())

def closed_forms(r):
    lam1 = raw_moment(1) * CVOL / SVOL
    lam2 = (raw_moment(2) - 0.5 * raw_moment(3)
            + raw_moment(4) / (3 * np.pi) - raw_moment(5) / (40 * np.pi)) / SVOL
    var_lc = lam2 - lam1 ** 2
    q_lead = np.pi * r * LBAR ** 2 * CVOL / SVOL ** 2
    q_fin = q_lead * (1 + 8 * r / LBAR)                 # Appendix D end term
    zeta1 = (np.pi * r * LBAR / SVOL) ** 2 * var_lc     # leading order
    zeta1_fin = zeta1 * (1 + 8 * r / LBAR) ** 2         # finite-r enhancement
    return dict(lam1=lam1, lam2=lam2, var_lc=var_lc,
                q_lead=q_lead, q_fin=q_fin, zeta1=zeta1, zeta1_fin=zeta1_fin)

def nstar(r):
    return CVOL / (2 * np.pi * r * closed_forms(r)["var_lc"])

def var_components(n, r, finite=True):
    cf = closed_forms(r); q = cf["q_fin"]
    z1 = cf["zeta1_fin"] if finite else cf["zeta1"]
    s2 = q * (1 - q) - 2 * z1
    vp = n * (n - 1) * (n - 2) * z1
    vr = (n * (n - 1) / 2) * s2
    return vp, vr, vp + vr

def iso_dirs(n, rng):
    v = rng.normal(size=(n, 3)); v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v

def oblate_dirs(n, rng, c=0.5):
    v = iso_dirs(n, rng); v[:, 2] *= c
    v /= np.linalg.norm(v, axis=1, keepdims=True)
    return v

def seg_seg(P1, P2, Q1, Q2, want_mid=False):
    """Minimum distance between segments [P1,P2] and [Q1,Q2] (clamped
    Lumelsky scheme), vectorized over rows; optionally the closest-point
    midpoint."""
    d1, d2, r = P2 - P1, Q2 - Q1, P1 - Q1
    a = (d1 * d1).sum(1); e = (d2 * d2).sum(1); f = (d2 * r).sum(1)
    b = (d1 * d2).sum(1); c = (d1 * r).sum(1)
    den = a * e - b * b
    s = np.where(den > 1e-12, np.clip((b * f - c * e) / den, 0, 1), 0.0)
    t = np.clip((b * s + f) / e, 0, 1)
    s = np.clip((b * t - c) / a, 0, 1)
    cpA, cpB = P1 + s[:, None] * d1, Q1 + t[:, None] * d2
    dist = np.linalg.norm(cpA - cpB, axis=1)
    if want_mid:
        return dist, 0.5 * (cpA + cpB)
    return dist

def sample_fibers(n, rng, dirfun=None):
    """Enlarged seeding: midpoints uniform on S; returns segment endpoints."""
    L = rng.choice(ELL, size=n, p=PMIX)
    X = rng.uniform(-LMAX / 2, 1 + LMAX / 2, size=(n, 3))
    v = iso_dirs(n, rng) if dirfun is None else dirfun(n, rng)
    return X - 0.5 * L[:, None] * v, X + 0.5 * L[:, None] * v

def count_once(n, twor, rng, dirfun=None):
    """One realization: enlarged seeding + contact-point-in-C gate."""
    P1, P2 = sample_fibers(n, rng, dirfun)
    i, j = np.triu_indices(n, 1)
    dist, mid = seg_seg(P1[i], P2[i], P1[j], P2[j], want_mid=True)
    inC = np.all((mid >= 0) & (mid <= 1), axis=1)
    return int(np.sum((dist <= twor) & inC))

# === 06_coverage_and_estimation.ipynb ======================================
# Reuses the counts from notebook 04 (../data/normality_counts.npz).
# (a) Coverage of closed-form-standardized normal intervals (centering by
#     E[N] and scaling by the finite-r Var(N)), with raw / continuity-corrected
#     Kolmogorov distance. (b) Method-of-moments estimation of r from one
#     observed count, with a delta-method 95% interval and its coverage.
r = 0.01; cf = closed_forms(r); q = cf["q_fin"]
d = np.load(os.path.join(_DAT, "normality_counts.npz"))
ns = sorted(int(k) for k in d.files)
z90, z95, z99 = stats.norm.ppf([0.95, 0.975, 0.995])

def mean_pred(n): return (n * (n - 1) / 2) * q
def var_pred(n):  return var_components(n, r, finite=True)[2]

crows = []
for n in ns:
    c = d[str(n)]; R = len(c)
    mu, sd = mean_pred(n), np.sqrt(var_pred(n))
    zc = (c - mu) / sd
    cov = lambda zz: float(np.mean(np.abs(zc) <= zz))
    up = float(np.mean(zc <= z95))
    mu_h, sd_h = c.mean(), c.std(ddof=1); zs = (c - mu_h) / sd_h
    xs = np.sort(zs); dK_raw = float(np.max(np.abs(np.arange(1, R + 1) / R - stats.norm.cdf(xs))))
    ks = np.sort(np.unique(c)); Fhat = np.array([np.mean(c <= k) for k in ks])
    dK_cc = float(np.max(np.abs(Fhat - stats.norm.cdf((ks + 0.5 - mu_h) / sd_h))))
    crows.append(dict(n=n, two90=cov(z90), two95=cov(z95), two99=cov(z99),
                      upper95=up, dK_raw=dK_raw, dK_cc=dK_cc))
ctab = pd.DataFrame(crows)
save_table(ctab, "table_coverage"); save_csv(ctab, "coverage")
print("Coverage (closed-form standardized):"); print(ctab.to_string(index=False))

# --- method-of-moments estimator for r -----------------------------------
Kof = lambda n: (n * (n - 1) / 2) * np.pi * LBAR ** 2 * CVOL / SVOL ** 2
def rhat(N, n):
    K = Kof(n); A = K * 8 / LBAR
    return (-K + np.sqrt(K * K + 4 * A * N)) / (2 * A)
def dEdr(n, rr): return Kof(n) * (1 + 16 * rr / LBAR)
zc95 = stats.norm.ppf(0.975); r_true = 0.01
erows = []
for n in ns:
    if n < 400: continue
    c = d[str(n)]
    rh = np.array([rhat(N, n) for N in c])
    se = np.array([np.sqrt(var_components(n, ri, finite=True)[2]) / dEdr(n, ri) for ri in rh])
    cover = float(np.mean((rh - zc95 * se <= r_true) & (r_true <= rh + zc95 * se)))
    erows.append(dict(n=n, rhat_mean=rh.mean(), bias=rh.mean() - r_true,
                      emp_sd=rh.std(ddof=1), delta_se=se.mean(), coverage95=cover))
etab = pd.DataFrame(erows)
save_table(etab, "table_estimator", float_format="%.6f"); save_csv(etab, "estimator")
print("\nMethod-of-moments estimation of r (true r = 0.01):")
print(etab.to_string(index=False))

fig, ax = plt.subplots(figsize=(4.6, 3.4))
nom = np.linspace(0.5, 0.999, 50)
for n, style in [(ns[0], ("o", GRAY)), (ns[-1], ("s", DARK))]:
    c = d[str(n)]; zc = (c - mean_pred(n)) / np.sqrt(var_pred(n))
    emp = [np.mean(np.abs(zc) <= stats.norm.ppf(0.5 + p / 2)) for p in nom]
    ax.plot(nom, emp, style[0], color=style[1], ms=3, label=f"n={n}")
ax.plot([0.5, 1], [0.5, 1], ls="--", color="0.6")
ax.set_xlabel("nominal two-sided coverage"); ax.set_ylabel("empirical coverage")
ax.legend(frameon=False, fontsize=8)
savefig(fig, "fig_coverage")

Coverage (closed-form standardized):
   n    two90    two95    two99  upper95   dK_raw    dK_cc
 200 0.908333 0.949167 0.984167 0.957500 0.090497 0.035130
 400 0.883750 0.951250 0.985000 0.965000 0.044305 0.023822
 800 0.882500 0.945000 0.992500 0.962500 0.052923 0.042992
1600 0.893333 0.920000 0.993333 0.946667 0.037125 0.039357
3200 0.933333 0.966667 1.000000 0.966667 0.086769 0.088643

Method-of-moments estimation of r (true r = 0.01):
   n  rhat_mean     bias   emp_sd  delta_se  coverage95
 400   0.010008 0.000008 0.001730  0.001682    0.931250
 800   0.010090 0.000090 0.000994  0.000991    0.950000
1600   0.010141 0.000141 0.000608  0.000618    0.940000
3200   0.010017 0.000017 0.000361  0.000400    0.966667
